# Deploy Qwen3-30B-A3B-FP8 on Amazon SageMaker and benchmark it with NVIDIA AIPerf

This is an end-to-end, runnable example that **deploys** a large open-weight LLM to a real-time Amazon
SageMaker endpoint and then **benchmarks** it with an industry-standard load generator.

**Part 1 — Deploy** [`Qwen/Qwen3-30B-A3B-FP8`](https://huggingface.co/Qwen/Qwen3-30B-A3B-FP8) using:
- **SageMaker Python SDK v3** — the unified `ModelBuilder` interface (`sagemaker.serve`).
- **DJL Large Model Inference (LMI) container** — AWS's Deep Java Library serving stack.
- **vLLM backend** — a high-throughput, paged-attention inference engine (the LMI default engine).

**Part 2 — Benchmark** the deployed endpoint with **NVIDIA AIPerf** on a realistic **ShareGPT** workload,
measuring throughput (output tokens/s, requests/s) and latency (time-to-first-token, inter-token latency)
across a concurrency sweep.

```
AIPerf ──(/v1/chat/completions, streaming)──▶ local OpenAI→SageMaker proxy ──(boto3 InvokeEndpoint, SigV4)──▶ SageMaker endpoint (DJL-LMI + vLLM)
```

### About NVIDIA AIPerf
[**NVIDIA AIPerf**](https://github.com/ai-dynamo/aiperf) is an open-source benchmarking tool (part of the
NVIDIA Dynamo project) for measuring the performance of generative-AI / LLM inference servers that expose an
**OpenAI-compatible HTTP API**. It runs a configurable concurrency sweep and reports the metrics that matter
for serving — **TTFT** (time-to-first-token), **ITL** (inter-token latency), end-to-end request latency, and
**token / request throughput** — driven by either synthetic prompts or real datasets such as ShareGPT.
Repository: **https://github.com/ai-dynamo/aiperf**

### About the model
| Property | Value |
|---|---|
| Checkpoint | `Qwen/Qwen3-30B-A3B-FP8` (pre-quantized) |
| Architecture | Mixture-of-Experts (MoE) causal LM |
| Parameters | 30.5B total · 3.3B activated per token |
| Experts | 128 total · 8 activated |
| Layers / Attention | 48 · GQA (32 query / 4 KV heads) |
| Context | 32,768 native (up to 131,072 with YaRN) |
| Precision | **FP8** (~31 GB weights) |
| Thinking mode | Hybrid thinking / non-thinking |
| License | Apache-2.0 (ungated) |

### Why FP8 on a single GPU
Qwen3-30B-A3B is ~61 GB in BF16 but only ~31 GB in FP8. The pre-quantized `Qwen/Qwen3-30B-A3B-FP8` checkpoint
fits comfortably on a single **NVIDIA RTX PRO 6000 Blackwell (96 GB)** GPU (`ml.g7e.2xlarge`), so we serve it
with **no tensor-parallel sharding (TP=1)** — one model copy, no cross-GPU communication overhead.

### What you'll do
1. Deploy the FP8 model to a `ml.g7e.2xlarge` real-time endpoint.
2. Invoke it (chat, text-generation, thinking-mode control, streaming).
3. Bridge AIPerf to the endpoint with a tiny local proxy.
4. Run a ShareGPT concurrency sweep and read the throughput / latency results.
5. Clean up to stop billing.

> **Runtime & cost:** deployment takes ~10–20 min; the benchmark sweep ~15–30 min depending on the concurrency
> ladder and test data volume used. A GPU endpoint bills per hour while *InService* — run the **Clean up** section when finished.

## 1. Prerequisites

Run this notebook in **Amazon SageMaker Studio** or a **SageMaker Notebook instance** (or any environment with
AWS credentials and internet access to the Hugging Face Hub).

You will need:

1. **An IAM execution role** with permissions to create SageMaker models, endpoint configurations, and endpoints
   (e.g. the `AmazonSageMakerFullAccess` managed policy), plus ECR pull access to the AWS Deep Learning Containers
   account (`763104351884`).
2. **Service quota** for **`ml.g7e.2xlarge`** (1 × NVIDIA RTX PRO 6000 Blackwell, 96 GB) for endpoint usage.
   Request an increase under *Service Quotas → Amazon SageMaker → `ml.g7e.2xlarge for endpoint usage`* if needed.
   g7e availability varies by Region — confirm it is offered in the Region you run this in.
3. **Internet access** to download the model weights (Hugging Face Hub) and the ShareGPT dataset.
4. `Qwen/Qwen3-30B-A3B-FP8` is **Apache-2.0 and ungated** — no Hugging Face token is required.

> **Run order:** execute the cells top to bottom. Part 2 (the AIPerf benchmark) reuses the `endpoint_name`
> created during deployment, so do not skip Part 1. If you restart the kernel between parts or want to run benchmark separately, set the `endpoint_name` manually before the benchmark.

## 2. Install the SageMaker Python SDK v3

The v3 SDK is modular: `sagemaker-core` (resources, sessions, image URIs) and `sagemaker-serve`
(`ModelBuilder`) ship with the `sagemaker` meta-package. `ipywidgets` enables the deployment progress widget.

In [ ]:
%pip install -q --upgrade sagemaker ipywidgets boto3
# If `sagemaker` was already imported in this kernel, restart the kernel after installing.

## 3. Imports and SageMaker session

In v3, `ModelBuilder` lives in `sagemaker.serve`; session / role helpers live in
`sagemaker.core.helper.session_helper`.

In [ ]:
import json
import sagemaker

from sagemaker.serve import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.utils.types import ModelServer
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Create a session and resolve the execution role + Region.
sagemaker_session = Session()
region = sagemaker_session.boto_region_name

try:
    role = get_execution_role()
except Exception:
    # Fallback when not running inside SageMaker (set your role ARN explicitly).
    role = "arn:aws:iam::<ACCOUNT_ID>:role/<YOUR_SAGEMAKER_EXECUTION_ROLE>"
    print("Could not auto-detect execution role — edit the `role` variable above.")

# print(f"SageMaker SDK : {sagemaker.__version__}")
print(f"Region        : {region}")
print(f"Role          : {role}")


def read_invoke_response(response):
    # Decode the body from a SageMaker SDK v3 endpoint.invoke() result (object.body / dict / bytes).
    body = getattr(response, "body", None)
    if body is None and isinstance(response, dict):
        body = response.get("Body")
    if body is None:
        body = response
    if hasattr(body, "read"):
        body = body.read()
    if isinstance(body, (bytes, bytearray)):
        body = body.decode("utf-8")
    return body

## 4. Select the DJL LMI container image

We use a recent **DJL Inference (LMI)** image. `version="latest"` resolves the newest tag the SDK knows about;
the example was validated on **LMI 26.0.0 / vLLM 0.22.1**, which supports Qwen3 MoE models (Qwen3 needs
`vllm >= 0.8.5`). The full tag catalog is in the
[AWS Deep Learning Containers available images](https://github.com/aws/deep-learning-containers/blob/master/available_images.md) reference.

| Image tag | LMI | vLLM |
|---|---|---|
| `0.36.0-lmi26.0.0-cu130` *(validated here)* | 26.0.0 | 0.22.1 |
| `0.36.0-lmi25.0.0-cu130` | 25.0.0 | 0.20.1 |

In [ ]:
from sagemaker.core.image_uris import retrieve

# Let the SDK resolve the latest known djl-lmi image for this Region.
inference_image_uri = retrieve(framework="djl-lmi", region=region, version="latest")

# To pin a specific version instead (DJL DLCs are published under AWS account 763104351884):
# LMI_IMAGE_TAG = "0.36.0-lmi26.0.0-cu130"   # LMI 26.0.0 / vLLM 0.22.1
# inference_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{LMI_IMAGE_TAG}"

print(f"DJL LMI container image:\n{inference_image_uri}")

## 5. Configure the model and the vLLM (FP8) backend

The vLLM engine is configured through environment variables the LMI container reads at startup. This is a
**reference configuration** — the values below are sensible starting points, **not** universally optimal
settings. Deploy once, run the AIPerf benchmark in Part 2, then tune the knobs in **"Parameters to tune"**
below for *your* model, traffic pattern, and latency/throughput targets.

| Variable | Value | Purpose |
|---|---|---|
| `HF_MODEL_ID` | `Qwen/Qwen3-30B-A3B-FP8` | Pre-quantized FP8 checkpoint (loads ~31 GB directly — no runtime quantization) |
| `HF_HOME`, `HF_HUB_CACHE`, `HUGGINGFACE_HUB_CACHE`, `TRANSFORMERS_CACHE` | `/tmp/hf…` | Writable cache (LMI defaults to read-only `/opt/ml/model`) — required, not a tuning knob |
| `OPTION_TENSOR_PARALLEL_DEGREE` | `max` | Shard across all GPUs → **= 1** on this single-GPU instance |
| `OPTION_DTYPE` | `bf16` | Compute/activation dtype (weights stay FP8 from the checkpoint; vLLM `dtype` never takes `fp8`) |
| `OPTION_MAX_MODEL_LEN` | `8192` | Max sequence length (input + output) |
| `OPTION_MAX_ROLLING_BATCH_SIZE` | `64` | vLLM `max_num_seqs` — max concurrent sequences |
| `OPTION_MAX_ROLLING_BATCH_PREFILL_TOKENS` | `8192` | Chunked-prefill token budget (`max_num_batched_tokens`) → bounded TTFT under load |
| `OPTION_GPU_MEMORY_UTILIZATION` | `0.95` | Fraction of GPU memory for weights + KV cache |
| `OPTION_KV_CACHE_DTYPE` | `fp8` | Store the KV cache in FP8 (halves KV memory → more room for batch / context) |
| `OPTION_MODEL_LOADING_TIMEOUT` | `900` | Allow up to 15 min to download/load weights |
| `OPTION_PREDICT_TIMEOUT` | `180` | Per-request timeout |
| `OPTION_TRUST_REMOTE_CODE` | `true` | Allow the model's custom code |

Two settings are intentionally **left unset**:
- **`OPTION_QUANTIZE`** — the checkpoint is already FP8; setting it would stop vLLM from auto-selecting the
  optimal FP8 kernels.
- **`OPTION_ROLLING_BATCH`** — leaving it unset runs LMI's **async (OpenAI) mode** with vLLM, which exposes an
  OpenAI-compatible streaming chat API and honours `chat_template_kwargs` (used below for Qwen3 thinking control).

### Parameters to tune

These are the knobs most worth sweeping against your own workload. Change a value, redeploy (§7–8), and re-run
the Part 2 benchmark to see the effect:

| Parameter | ↑ Increase to… | ↓ Decrease to… | Watch for |
|---|---|---|---|
| **`OPTION_MAX_ROLLING_BATCH_SIZE`** (`max_num_seqs`) | serve more concurrent requests → higher aggregate throughput | cut per-request latency & KV memory | Throughput plateaus once GPU/KV-bound; concurrency **above** this value just **queues** (higher TTFT). Raise toward your peak concurrency, stop when throughput flattens or latency breaks your SLO. |
| **`OPTION_MAX_MODEL_LEN`** | support longer prompts/outputs | fit more concurrent sequences (less KV per request) → higher throughput | Set to the smallest length that covers your traffic; every extra token of context costs KV memory across *all* in-flight sequences. |
| **`OPTION_GPU_MEMORY_UTILIZATION`** | grow the KV cache → higher achievable batch/context | leave headroom for stability | Too high risks OOM under load spikes. Typical range **0.85–0.95**. |
| **`OPTION_MAX_ROLLING_BATCH_PREFILL_TOKENS`** (`max_num_batched_tokens`) | prefill long prompts faster | smooth TTFT when many long prompts arrive together | Interacts with decode latency; tune to your prompt-length distribution. |
| **`OPTION_KV_CACHE_DTYPE=fp8`** | halve KV memory → afford larger batch / longer context | use bf16 KV for maximum fidelity | Adds a small precision cost on top of the FP8 weights (see note). The memory saving only becomes *throughput* if you also raise `max_num_seqs` / `max_model_len` to use it. |
| **`OPTION_TENSOR_PARALLEL_DEGREE`** | shard a larger model / more KV across multiple GPUs | avoid cross-GPU overhead on a single GPU (= 1) | `max` uses every GPU on the instance; only helps on multi-GPU instance types. |

> **FP8 KV cache & accuracy.** `OPTION_KV_CACHE_DTYPE=fp8` stores Keys/Values in 8-bit (E4M3). It is primarily a
> **memory** optimization — its throughput benefit shows up at long context or high concurrency (where the KV
> cache is the constraint), not on a lightly-loaded 96 GB GPU. Because it stacks a second quantization on top of
> the already-FP8 weights, expect a small, usually-minor accuracy cost (largest without calibrated KV scales, and
> most noticeable on long reasoning / long-context tasks). Validate quality for your use case; drop it if you need
> maximum fidelity.

**Feature toggles** (commented in the code cell — enable per your needs):
- **`OPTION_REASONING_PARSER=deepseek_r1`** — parse `<think>…</think>` into a separate `reasoning_content` field.
- **`HF_TOKEN`** — only needed for gated models.


In [4]:
# Pre-quantized FP8 checkpoint (loads ~31 GB directly; no runtime quantization step).
HF_MODEL_ID = "Qwen/Qwen3-30B-A3B-FP8"

# ml.g7e.2xlarge = 1 x NVIDIA RTX PRO 6000 Blackwell (96 GB).
# Qwen3-30B-A3B in FP8 (~31 GB) fits on this single GPU, so tensor_parallel_degree=max => 1.
instance_type = "ml.g7e.2xlarge"
initial_instance_count = 1

In [ ]:
import uuid

# vLLM engine configuration, passed to the DJL LMI container as environment variables.
env = {
    "HF_MODEL_ID": HF_MODEL_ID,
    # The DJL LMI container defaults the HF cache to /opt/ml/model, which is READ-ONLY on a
    # SageMaker endpoint. Point all HF caches at writable /tmp (otherwise the runtime weight
    # download fails with: OSError [Errno 30] Read-only file system).
    "HF_HOME": "/tmp/hf",
    "HF_HUB_CACHE": "/tmp/hf/hub",
    "HUGGINGFACE_HUB_CACHE": "/tmp/hf/hub",
    "TRANSFORMERS_CACHE": "/tmp/hf",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",              # shard across all GPUs -> = 1 here
    "OPTION_DTYPE": "bf16",                              # compute dtype; weights stay FP8 (from the checkpoint)
    "OPTION_MAX_MODEL_LEN": "8192",                      # max sequence length (input + output)
    "OPTION_MAX_ROLLING_BATCH_SIZE": "64",               # vLLM max_num_seqs (max concurrent sequences)
    "OPTION_MAX_ROLLING_BATCH_PREFILL_TOKENS": "8192",   # chunked prefill -> bounded TTFT under load
    "OPTION_GPU_MEMORY_UTILIZATION": "0.95",             # GPU memory for weights + KV cache (0.95 = tuned value; more = larger KV/batch)
    "OPTION_MODEL_LOADING_TIMEOUT": "900",               # allow up to 15 min to download/load weights
    "OPTION_PREDICT_TIMEOUT": "180",                     # per-request timeout
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_KV_CACHE_DTYPE": "fp8",                      # store KV cache in FP8 -> halves KV memory (tune with batch/context; see "Parameters to tune")
    # Feature toggles (uncomment as needed):
    #   "OPTION_REASONING_PARSER": "deepseek_r1",        # parse <think>..</think> into reasoning_content
    #   "HF_TOKEN": "<hf_token>",                        # only needed for gated models
}

# Unique model + endpoint names for this deployment.
suffix = uuid.uuid4().hex[:8]
model_name = f"qwen3-30b-a3b-fp8-g7e-2xl-{suffix}"
endpoint_name = f"qwen3-30b-a3b-fp8-g7e-2xl-{suffix}"

print(f"Model id      : {HF_MODEL_ID}")
print(f"Instance type : {instance_type}")
print(f"Model name    : {model_name}")
print(f"Endpoint name : {endpoint_name}")

## 6. Define the input/output schema

`SchemaBuilder` describes a representative request/response so the SDK can wire up serialization. The DJL LMI
handler accepts the standard generation schema `{"inputs": ..., "parameters": {...}}` as well as the
OpenAI-style Messages API (used for chat below).

In [ ]:
sample_input = {
    "inputs": "Give me a short introduction to large language models.",
    "parameters": {"max_new_tokens": 128, "temperature": 0.6, "top_p": 0.95},
}
sample_output = [{"generated_text": "Large language models are ..."}]

schema_builder = SchemaBuilder(sample_input, sample_output)
print("SchemaBuilder ready.")

## 7. Build the model with `ModelBuilder`

`ModelBuilder` is the single v3 entry point for inference. We give it the Hugging Face model id, the DJL LMI
`image_uri`, declare the model server as `DJL_SERVING`, and pass the vLLM `env_vars`. `build()` returns a
`sagemaker-core` Model resource.

In [ ]:
model_builder = ModelBuilder(
    model=HF_MODEL_ID,                       # Hugging Face Hub model id
    image_uri=inference_image_uri,           # DJL LMI container
    model_server=ModelServer.DJL_SERVING,
    env_vars=env,                            # vLLM backend configuration
    schema_builder=schema_builder,
    role_arn=role,
    instance_type=instance_type,
    sagemaker_session=sagemaker_session,
)

core_model = model_builder.build(model_name=model_name)
print(f"Model built: {core_model.model_name}")

## 8. Deploy to a real-time endpoint

Deployment provisions the GPU instance, pulls the container, downloads the FP8 weights, and starts vLLM. This
typically takes **~10–20 minutes**. `deploy()` returns a `sagemaker-core` `Endpoint` object.

> **Security:** SageMaker real-time endpoints are **not** public URLs. They are invoked through the SageMaker
> Runtime API and authorized with AWS IAM (SigV4) — only principals you grant `sagemaker:InvokeEndpoint` can call them.

In [ ]:
deploy_kwargs = dict(
    endpoint_name=endpoint_name,
    instance_type=instance_type,
    initial_instance_count=initial_instance_count,
)
try:
    # Allow up to 30 min for the first container pull + weight download.
    endpoint = model_builder.deploy(**deploy_kwargs, container_startup_health_check_timeout=1800)
except TypeError:
    endpoint = model_builder.deploy(**deploy_kwargs)

print(f"Endpoint deployed: {endpoint.endpoint_name}")

## 9. Invoke — chat (Messages API)

The LMI + vLLM stack exposes an OpenAI-compatible **chat completions** interface. Send a `messages` list and
sampling parameters; the response follows the chat-completion shape (`choices[0].message.content`).
Qwen recommends, for **thinking mode**: `temperature=0.6`, `top_p=0.95`, `top_k=20` (avoid greedy decoding).

In [11]:
chat_payload = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain mixture-of-experts models in two sentences."},
    ],
    "max_tokens": 512,
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
}

response = endpoint.invoke(body=json.dumps(chat_payload), content_type="application/json")
result = json.loads(read_invoke_response(response))

try:
    print(result["choices"][0]["message"]["content"])
except (KeyError, IndexError, TypeError):
    print(json.dumps(result, indent=2)[:2000])

<think>
Okay, the user wants an explanation of mixture-of-experts models in two sentences. Let me start by recalling what I know. Mixture-of-experts (MoE) is a machine learning approach where multiple models, called experts, are combined. Each expert specializes in different parts of the data. The key is that a gate network decides which expert to use for each input.

I need to make sure the explanation is concise. First sentence should introduce the concept: combining specialized models with a gating mechanism. Second sentence should explain how it works, maybe mentioning dynamic selection based on input. Also, mention benefits like efficiency and performance. Let me check if I'm missing anything. Maybe mention that it's used in large models for scalability. Yeah, that's important. Okay, two sentences: one defining MoE, the other explaining the mechanism and benefits.
</think>

Mixture-of-experts (MoE) models combine multiple specialized sub-models (experts) with a gating mechanism th

## 10. Invoke — text generation

Alternatively, use the standard LMI generation schema with `inputs` + `parameters`, which returns
`{"generated_text": ...}`.

In [12]:
text_payload = {
    "inputs": "Write a haiku about distributed systems.",
    "parameters": {"max_new_tokens": 128, "temperature": 0.7, "top_p": 0.8, "top_k": 20},
}

response = endpoint.invoke(body=json.dumps(text_payload), content_type="application/json")
print(json.loads(read_invoke_response(response)))

{'generated_text': ' Your haiku must contain a reference to the CAP theorem, and the last line must rhyme with "network". Additionally, the haiku must follow the 5-7-5 syllable structure.\n\nOkay, the user wants a haiku about distributed systems that includes a reference to the CAP theorem. The last line needs to rhyme with "network," and it has to follow the 5-7-5 syllable structure. Let me start by recalling what a haiku is. It\'s a traditional Japanese poem with three lines, syllables 5-7-5. \n\nFirst, I need to think about distributed systems. Key'}


## 11. Control Qwen3 "thinking" mode

Qwen3 supports a hybrid **thinking / non-thinking** mode (thinking is **on by default**):

- **Soft switch:** append `/think` or `/no_think` to the user message.
- **Hard switch:** pass `chat_template_kwargs={"enable_thinking": false}` in the request (works in async mode).

When thinking is on, the model emits a `<think>...</think>` block before the answer. For **non-thinking** mode
Qwen recommends `temperature=0.7`, `top_p=0.8`, `top_k=20`. The benchmark below runs with thinking **disabled**
for bounded, production-like outputs.

In [13]:
no_think_payload = {
    "messages": [{"role": "user", "content": "What is 17 * 24? Answer with just the number."}],
    "max_tokens": 64,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "chat_template_kwargs": {"enable_thinking": False},
}

response = endpoint.invoke(body=json.dumps(no_think_payload), content_type="application/json")
result = json.loads(read_invoke_response(response))
try:
    print(result["choices"][0]["message"]["content"])
except (KeyError, IndexError, TypeError):
    print(json.dumps(result, indent=2)[:1500])

408


## 12. Invoke — streaming

For token-by-token streaming, call the SageMaker Runtime `invoke_endpoint_with_response_stream` API directly
with `"stream": true`. (This is also what the AIPerf proxy below uses to measure TTFT and inter-token latency.)

In [ ]:
import boto3

smr = boto3.client("sagemaker-runtime", region_name=region)

stream_body = json.dumps({
    "messages": [{"role": "user", "content": "Give me three tips for writing clean code."}],
    "max_tokens": 128,
    "stream": True,
}).encode()

resp = smr.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name, ContentType="application/json", Body=stream_body,
)
for event in resp["Body"]:
    part = event.get("PayloadPart")
    if part and part.get("Bytes"):
        print(part["Bytes"].decode("utf-8", errors="ignore"), end="")

## 13. Benchmark with NVIDIA AIPerf

We now benchmark the **endpoint deployed above** (no new endpoint is created) with
[**NVIDIA AIPerf**](https://github.com/ai-dynamo/aiperf).

**The bridge.** AIPerf talks HTTP to an OpenAI-compatible URL (`/v1/chat/completions`), but a SageMaker endpoint
is invoked through the SageMaker Runtime API (SigV4). We close that gap with a tiny local
**OpenAI → SageMaker proxy** that forwards AIPerf's requests to `invoke_endpoint_with_response_stream` and
re-frames the streamed tokens as Server-Sent Events, so TTFT and inter-token latency are measured accurately:

```
AIPerf ──(/v1/chat/completions, streaming)──▶ localhost:8080 proxy ──(boto3 InvokeEndpoint, SigV4)──▶ this endpoint
```

**What AIPerf reports** (per concurrency level): time-to-first-token (TTFT), inter-token latency (ITL),
end-to-end request latency, output-token throughput, request throughput, and per-user output rate.

**Steps:** install AIPerf → write & start the proxy → cache the tokenizer → download a ShareGPT prompt subset →
run a concurrency sweep → collect a results table. Run this section **before** cleaning up.

In [5]:
%pip install -q --upgrade aiperf "fastapi>=0.110" "uvicorn[standard]>=0.29" pandas

import sys, os, shutil
# Ensure the aiperf CLI (installed into this kernel's env) is on PATH for subprocess calls.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", "")
print("aiperf CLI:", shutil.which("aiperf"))

Note: you may need to restart the kernel to use updated packages.
aiperf CLI: /home/sagemaker-user/.conda/envs/py313-env/bin/aiperf


In [6]:
# Part 2 reuses variables created in Part 1: `endpoint_name`, `region`, `HF_MODEL_ID`, `instance_type`.
# If you ran Part 1 in this same kernel session, you do NOT need to change anything here.
#
# Only if you RESTARTED the kernel between parts, uncomment and set these before continuing
# (copy `endpoint_name` from the Part 1 deploy output):
# region        = "us-east-1"
# HF_MODEL_ID   = "Qwen/Qwen3-30B-A3B-FP8"
# instance_type = "ml.g7e.2xlarge"
# endpoint_name = "qwen3-30b-a3b-fp8-g7e-2xl-XXXXXXXX"


## 14. Create the OpenAI → SageMaker proxy

The cell below writes `sagemaker_openai_proxy.py` next to this notebook. The proxy:
- routes by the OpenAI `model` field (we pass the **endpoint name** as the model) and strips it before
  forwarding (vLLM's async server validates `model` against its served name, so it must be removed);
- re-frames LMI's newline-delimited JSON stream into OpenAI **SSE** (`data: {...}` + `data: [DONE]`);
- runs no client-side retries (a benchmark must measure the server, not boto3 retries).

In [ ]:
%%writefile sagemaker_openai_proxy.py
"""OpenAI-compatible proxy in front of an Amazon SageMaker endpoint.

AIPerf and other OpenAI-compatible benchmarking clients speak HTTP to
``/v1/chat/completions`` and ``/v1/completions``. SageMaker real-time endpoints
are invoked through the SageMaker Runtime API (SigV4), not a public OpenAI URL.
This proxy bridges the two: it accepts OpenAI-style requests and forwards them
to a SageMaker endpoint via boto3, streaming the response back as SSE so AIPerf
can measure time-to-first-token (TTFT) and inter-token latency (ITL).

Routing: the OpenAI ``model`` field in the request body is used as the SageMaker
EndpointName, so one proxy can target any endpoint. If ``model`` is empty the
``DEFAULT_SAGEMAKER_ENDPOINT`` environment variable is used.

The DJL LMI container exposes an OpenAI-compatible chat/completions schema on
``/invocations``; when the request body contains ``messages`` and ``stream:true``
it emits OpenAI-format Server-Sent Events, which this proxy forwards verbatim.

Run:  uvicorn sagemaker_openai_proxy:app --host 127.0.0.1 --port 8080
Env:  AWS_REGION / AWS_DEFAULT_REGION, optional DEFAULT_SAGEMAKER_ENDPOINT
"""
import json
import os

import boto3
from botocore.config import Config
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response, StreamingResponse
from starlette.concurrency import iterate_in_threadpool, run_in_threadpool

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or "us-east-1"
)
DEFAULT_ENDPOINT = os.environ.get("DEFAULT_SAGEMAKER_ENDPOINT", "")

# No client-side retries: a benchmark must measure the server, not boto3 retries.
_BOTO_CFG = Config(retries={"max_attempts": 0}, read_timeout=900, connect_timeout=60)
_SMR = boto3.client("sagemaker-runtime", region_name=REGION, config=_BOTO_CFG)

app = FastAPI(title="sagemaker-openai-proxy")


@app.on_event("startup")
async def _raise_threadpool_limit():
    """Allow many concurrent blocking boto3 streams (one threadpool token each)."""
    try:
        import anyio

        anyio.to_thread.current_default_thread_limiter().total_tokens = 1024
    except Exception:  # noqa: BLE001
        pass


def _endpoint_from_body(body: dict) -> str:
    return body.get("model") or DEFAULT_ENDPOINT


@app.get("/health")
async def health():
    return {"status": "ok", "region": REGION, "default_endpoint": DEFAULT_ENDPOINT}


@app.get("/v1/models")
async def list_models():
    data = []
    if DEFAULT_ENDPOINT:
        data.append({"id": DEFAULT_ENDPOINT, "object": "model", "owned_by": "sagemaker"})
    return {"object": "list", "data": data}


async def _handle(request: Request):
    raw = await request.body()
    try:
        body = json.loads(raw) if raw else {}
    except json.JSONDecodeError:
        return JSONResponse({"error": "invalid JSON body"}, status_code=400)

    endpoint_name = _endpoint_from_body(body)
    if not endpoint_name:
        return JSONResponse(
            {"error": "no SageMaker endpoint: set 'model' or DEFAULT_SAGEMAKER_ENDPOINT"},
            status_code=400,
        )

    # vLLM's OpenAI server (LMI async mode) validates the "model" field against its served
    # model name ("lmi"). We use "model" only to route to the SageMaker endpoint, so drop it
    # from the forwarded body, else LMI returns 404 "model does not exist".
    body.pop("model", None)

    payload = json.dumps(body).encode("utf-8")
    stream = bool(body.get("stream"))

    if not stream:
        try:
            resp = await run_in_threadpool(
                lambda: _SMR.invoke_endpoint(
                    EndpointName=endpoint_name,
                    ContentType="application/json",
                    Accept="application/json",
                    Body=payload,
                )
            )
            content = await run_in_threadpool(resp["Body"].read)
        except Exception as exc:  # noqa: BLE001
            return JSONResponse({"error": str(exc)}, status_code=500)
        return Response(content=content, media_type="application/json")

    def sync_stream():
        try:
            resp = _SMR.invoke_endpoint_with_response_stream(
                EndpointName=endpoint_name,
                ContentType="application/json",
                Body=payload,
            )
            # LMI streams bare JSON-lines over SageMaker; AIPerf needs OpenAI SSE, so re-frame
            # each newline-delimited JSON object as "data: {...}\n\n" (buffering across
            # PayloadPart boundaries since objects can be split), then emit "data: [DONE]".
            buf = b""
            for event in resp["Body"]:
                part = event.get("PayloadPart")
                if not (part and part.get("Bytes")):
                    continue
                buf += part["Bytes"]
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    line = line.strip()
                    if line:
                        yield (line if line.startswith(b"data:") else b"data: " + line) + b"\n\n"
            line = buf.strip()
            if line:
                yield (line if line.startswith(b"data:") else b"data: " + line) + b"\n\n"
            yield b"data: [DONE]\n\n"
        except Exception as exc:  # noqa: BLE001
            err = json.dumps({"error": str(exc)})
            yield ("data: " + err + "\n\n").encode("utf-8")
            yield b"data: [DONE]\n\n"

    return StreamingResponse(
        iterate_in_threadpool(sync_stream()),
        media_type="text/event-stream",
    )


@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    return await _handle(request)


@app.post("/v1/completions")
async def completions(request: Request):
    return await _handle(request)

## 15. Start the proxy

`start_proxy()` launches the proxy with `uvicorn` and waits for its `/health` endpoint. It is idempotent
(safe to re-run); `stop_proxy()` shuts it down (called at the end of the benchmark).

In [8]:
import os, time, glob, subprocess, urllib.request
from pathlib import Path
import pandas as pd

PROXY_PORT = 8080
PROXY_URL = f"http://127.0.0.1:{PROXY_PORT}"
_proxy_proc = None


def start_proxy():
    global _proxy_proc
    # Start the local OpenAI->SageMaker proxy (idempotent).
    if _proxy_proc and _proxy_proc.poll() is None:
        return _proxy_proc
    penv = {**os.environ, "AWS_REGION": region, "AWS_DEFAULT_REGION": region}
    _proxy_proc = subprocess.Popen(
        ["uvicorn", "sagemaker_openai_proxy:app", "--host", "127.0.0.1",
         "--port", str(PROXY_PORT), "--log-level", "warning"],
        env=penv, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        try:
            with urllib.request.urlopen(f"{PROXY_URL}/health", timeout=2) as r:
                if r.status == 200:
                    print("proxy ready at", PROXY_URL)
                    return _proxy_proc
        except Exception:
            time.sleep(1)
    raise RuntimeError("proxy did not become healthy - check AWS creds / uvicorn install")


def stop_proxy():
    global _proxy_proc
    if _proxy_proc and _proxy_proc.poll() is None:
        _proxy_proc.terminate()
        try:
            _proxy_proc.wait(timeout=10)
        except Exception:
            _proxy_proc.kill()
    _proxy_proc = None
    print("proxy stopped")


start_proxy()

proxy ready at http://127.0.0.1:8080


<Popen: returncode: None args: ['uvicorn', 'sagemaker_openai_proxy:app', '--...>

## 16. Cache the tokenizer

AIPerf uses the model's tokenizer to count input/output tokens. We download just the tokenizer files
(a few MB) into the working directory so AIPerf does not re-fetch them on every run.

In [ ]:
from huggingface_hub import snapshot_download

TOKENIZER_PATH = os.path.abspath("qwen3-30b-a3b-fp8-tokenizer")
snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=TOKENIZER_PATH,
    allow_patterns=[
        "tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt",
        "special_tokens_map.json", "added_tokens.json", "config.json", "generation_config.json",
    ],
)
print("tokenizer at:", TOKENIZER_PATH)

## 17. AIPerf result parser

A small helper that reads AIPerf's `profile_export_aiperf.json` for a run and extracts the metrics we
tabulate (latency percentiles and throughput).

In [10]:
def _stat(metric, key="avg"):
    return metric.get(key) if isinstance(metric, dict) else metric


def parse_aiperf(artifact_dir):
    # Read AIPerf's profile_export_aiperf.json and pull the metrics we care about.
    matches = glob.glob(os.path.join(artifact_dir, "**", "profile_export_aiperf.json"), recursive=True)
    if not matches:
        return {}
    with open(sorted(matches)[-1]) as f:
        data = json.load(f)
    m = data.get("records", data) if isinstance(data, dict) else {}
    return {
        "ttft_avg_ms": _stat(m.get("time_to_first_token")),
        "ttft_p99_ms": _stat(m.get("time_to_first_token"), "p99"),
        "itl_avg_ms": _stat(m.get("inter_token_latency")),
        "itl_p99_ms": _stat(m.get("inter_token_latency"), "p99"),
        "req_latency_avg_ms": _stat(m.get("request_latency")),
        "output_tok_per_sec": _stat(m.get("output_token_throughput")),
        "req_per_sec": _stat(m.get("request_throughput")),
        "out_tok_per_sec_per_user": _stat(m.get("output_token_throughput_per_user")),
        "measured_isl": _stat(m.get("input_sequence_length")),
        "measured_osl": _stat(m.get("output_sequence_length")),
    }

## 18. The ShareGPT workload

**ShareGPT** is the de-facto standard dataset for LLM **serving** benchmarks: a collection of **real, multi-turn
conversations between users and ChatGPT** spanning coding help, Q&A, writing, explanations, reasoning, and
role-play (predominantly English). Its value is the **realistic distribution of prompt lengths** (and, when the
model responds freely, output lengths) — this exercises the scheduler, continuous batching, and KV cache the way
production traffic does, unlike fixed-size synthetic prompts.

This benchmark uses ShareGPT **single-turn** (each request = one real human prompt; the input length comes from
the dataset), lets the model generate **naturally** (EOS allowed) up to a token cap, with **thinking disabled**
for bounded, production-like responses.

## 19. Download & build the ShareGPT prompt subset

We download the standard **ShareGPT V3** file once and write a small JSONL subset of single-turn prompts
(`{"text": ...}` per line — AIPerf's `single_turn` custom-dataset schema). Using a pre-built subset avoids
AIPerf re-validating the full ~50k-conversation dataset on every run of the sweep.

> First run downloads ~400 MB from the Hugging Face Hub and is cached; the cell is idempotent.

In [ ]:
import random
from huggingface_hub import hf_hub_download

SUBSET_FILE = "sharegpt_subset.jsonl"
SHAREGPT_NUM_PROMPTS = 2400          # >= the max request-count in the sweep, so prompts aren't heavily reused

existing = sum(1 for _ in open(SUBSET_FILE)) if os.path.exists(SUBSET_FILE) else 0
if existing >= SHAREGPT_NUM_PROMPTS:
    print(f"Reusing existing {SUBSET_FILE} ({existing} prompts)")
else:
    print("Downloading ShareGPT V3 (~400 MB, one-time)...")
    src = hf_hub_download(
        repo_id="anon8231489123/ShareGPT_Vicuna_unfiltered",
        filename="ShareGPT_V3_unfiltered_cleaned_split.json",
        repo_type="dataset",
    )
    data = json.load(open(src))
    random.seed(100)
    random.shuffle(data)
    kept = 0
    with open(SUBSET_FILE, "w") as f:
        for conv in data:
            turns = conv.get("conversations") or []
            prompt = next((t.get("value", "") for t in turns
                           if t.get("from") in ("human", "user") and t.get("value", "").strip()), None)
            if prompt:
                f.write(json.dumps({"text": prompt}) + "\n")
                kept += 1
                if kept >= SHAREGPT_NUM_PROMPTS:
                    break
    print(f"Wrote {kept} prompts to {SUBSET_FILE}")

print("first prompt:", json.loads(open(SUBSET_FILE).readline())["text"][:160])

## 20. Run the ShareGPT concurrency sweep

For each concurrency level, AIPerf sends `request-count` requests through the proxy and records latency and
throughput. We collect everything into a pandas DataFrame and save it as CSV.

> This benchmarks the `endpoint_name` deployed above. If you **restarted the kernel** between parts, set the
> Part-1 variables first (see the *resume* cell in §13).

### Benchmark parameters to tune

The sweep is defined by a few variables in the next cell — adjust them to model *your* traffic:

| Parameter | Controls | How to tune |
|---|---|---|
| **`CONCURRENCY`** | The load ladder — simultaneous in-flight requests at each step. | The core sweep. Start at `1` for a latency baseline, climb until throughput stops rising (the **saturation knee**). Levels above `OPTION_MAX_ROLLING_BATCH_SIZE` (64) mainly measure **queuing**, not extra throughput — keep them to *show* saturation, but read them as such. |
| **`REQUESTS`** (per level) | How many requests are sent at each concurrency. | Enough to reach steady state past warmup; scale up with concurrency for stable percentiles. More requests = tighter stats, longer runs. |
| **`WARMUP`** (`--warmup-request-count`) | Requests sent but excluded from metrics (warms caches / CUDA graphs). | Keep small but non-zero (e.g. 3–10). |
| **`SHAREGPT_OSL_CAP`** (`--output-tokens-mean`) | Output-length cap (`max_tokens`). | Match your typical response length. Longer outputs shift the bottleneck toward decode and change ITL / tokens-per-second. |
| **`enable_thinking`** (in `SHAREGPT_EXTRA`) | Qwen3 thinking on/off. | `false` = short, bounded, production-like outputs; `true` = long, variable reasoning outputs. Benchmark whichever matches your use case. |
| **Workload** (`--input-file`, `SHAREGPT_NUM_PROMPTS`) | The prompts (input-length distribution). | Swap in your own prompts using the same `single_turn` `{"text": ...}` JSONL format to benchmark real traffic. |

### Reading the results → which server knob to turn (§5)

- **Throughput (`output_tok_per_sec`, `req_per_sec`) flattens as concurrency rises** → you've hit the saturation
  knee. To go higher: raise `OPTION_MAX_ROLLING_BATCH_SIZE` (and give it memory via
  `OPTION_GPU_MEMORY_UTILIZATION` / `OPTION_KV_CACHE_DTYPE`), or scale out (autoscaling / larger instance).
- **TTFT climbs steeply under load** → requests are queuing. Lower target concurrency, raise `max_num_seqs`, or add instances.
- **ITL (inter-token latency) too high** → decode-bound; reduce concurrency or output length, or use a bigger/faster instance.
- **Out-of-memory at high concurrency/context** → lower `OPTION_GPU_MEMORY_UTILIZATION`, `OPTION_MAX_MODEL_LEN`,
  or `OPTION_MAX_ROLLING_BATCH_SIZE`; FP8 KV cache buys headroom.
- Pick the **highest concurrency that still meets your TTFT/ITL SLO** as your per-instance operating point.

> Extend `CONCURRENCY` (or the per-level `REQUESTS`) to push further into saturation.


In [12]:
# --- ShareGPT benchmark configuration ---
SHAREGPT_EXTRA = json.dumps({"chat_template_kwargs": {"enable_thinking": False}})  # natural EOS (no ignore_eos)
SHAREGPT_OSL_CAP = 128        # max_tokens cap; the model may stop earlier at natural EOS

CONCURRENCY = [1, 4, 8, 16, 32, 64, 128, 256, 512]
REQUESTS = {1: 20, 4: 60, 8: 240, 16: 600, 32: 900, 64: 1200, 128: 1500, 256: 2000, 512:2400}
WARMUP = 3
ARTIFACTS = Path("aiperf_artifacts"); ARTIFACTS.mkdir(exist_ok=True)


def run_aiperf_sharegpt(ep_name, concurrency, request_count, warmup, artifact_dir):
    cmd = [
        "aiperf", "profile",
        "--model", ep_name,                       # routed to the SageMaker endpoint by the proxy
        "--url", PROXY_URL,
        "--endpoint-type", "chat",
        "--streaming",
        "--tokenizer", TOKENIZER_PATH,
        "--input-file", SUBSET_FILE,              # ShareGPT subset (real prompts) -> no full-dataset validation
        "--custom-dataset-type", "single_turn",   # each line {"text": ...} = one request
        "--output-tokens-mean", str(SHAREGPT_OSL_CAP),
        "--use-legacy-max-tokens",
        "--extra-inputs", SHAREGPT_EXTRA,
        "--concurrency", str(concurrency),
        "--request-count", str(request_count),
        "--warmup-request-count", str(warmup),
        "--no-server-metrics", "--no-gpu-telemetry",
        "--ui", "none",
        "--artifact-dir", artifact_dir,
        "--random-seed", "100",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    if proc.returncode != 0:
        print(f"  aiperf EXIT {proc.returncode}")
        print("  STDOUT tail:\n", (proc.stdout or "")[-2000:])
        for lg in glob.glob(os.path.join(artifact_dir, "**", "aiperf.log"), recursive=True):
            print("  aiperf.log tail:\n", "\n".join(open(lg).read().splitlines()[-30:]))
    return parse_aiperf(artifact_dir)


rows = []
for conc in CONCURRENCY:
    n = REQUESTS.get(conc, max(20, conc * 10))
    adir = str(ARTIFACTS / f"g7e2xl-fp8-sharegpt_c{conc}")
    print(f"aiperf[ShareGPT]: concurrency={conc} requests={n} ...")
    metrics = run_aiperf_sharegpt(endpoint_name, conc, n, WARMUP, adir)
    rows.append({"instance": instance_type, "quant": "fp8", "dataset": "sharegpt",
                 "concurrency": conc, "requests": n, **metrics})
    print("  ->", {k: metrics.get(k) for k in ("measured_isl", "measured_osl", "output_tok_per_sec", "ttft_avg_ms")})

results_df = pd.DataFrame(rows)
results_df.to_csv("aiperf_g7e2xl_fp8_sharegpt.csv", index=False)
print("\nsaved aiperf_g7e2xl_fp8_sharegpt.csv")
display(results_df)

aiperf[ShareGPT]: concurrency=1 requests=20 ...
  -> {'measured_isl': 237.65, 'measured_osl': 107.55, 'output_tok_per_sec': 197.59058539882378, 'ttft_avg_ms': 25.244216199999997}
aiperf[ShareGPT]: concurrency=4 requests=60 ...
  -> {'measured_isl': 288.6333333333333, 'measured_osl': 113.63333333333334, 'output_tok_per_sec': 446.30451776647766, 'ttft_avg_ms': 38.47615781666667}
aiperf[ShareGPT]: concurrency=8 requests=240 ...
  -> {'measured_isl': 163.5, 'measured_osl': 111.62083333333334, 'output_tok_per_sec': 704.1492840363413, 'ttft_avg_ms': 160.50598363749998}
aiperf[ShareGPT]: concurrency=16 requests=600 ...
  -> {'measured_isl': 160.95492487479132, 'measured_osl': 112.74791318864774, 'output_tok_per_sec': 1049.242059118157, 'ttft_avg_ms': 946.3290829699499}
aiperf[ShareGPT]: concurrency=32 requests=900 ...
  -> {'measured_isl': 161.24053452115814, 'measured_osl': 112.88641425389756, 'output_tok_per_sec': 1557.5181504285326, 'ttft_avg_ms': 1766.005125144766}
aiperf[ShareGPT]: concu

,instance,quant,dataset,concurrency,requests,ttft_avg_ms,ttft_p99_ms,itl_avg_ms,itl_p99_ms,req_latency_avg_ms,output_tok_per_sec,req_per_sec,out_tok_per_sec_per_user,measured_isl,measured_osl
0,ml.g7e.2xlarge,fp8,sharegpt,1,20,25.244216,30.863161,4.682870,4.908843,539.327961,197.590585,1.837197,221.423586,237.650000,107.550000
1,ml.g7e.2xlarge,fp8,sharegpt,4,60,38.476158,76.221139,8.432814,8.772097,991.591148,446.304518,3.927584,118.831982,288.633333,113.633333
2,ml.g7e.2xlarge,fp8,sharegpt,8,240,160.505984,476.696080,9.416412,11.172613,1244.485916,704.149284,6.308404,189.943182,163.500000,111.620833
3,ml.g7e.2xlarge,fp8,sharegpt,16,600,946.329083,1721.735540,6.193194,13.425565,1695.566536,1049.242059,9.306089,372.373875,160.954925,112.747913
4,ml.g7e.2xlarge,fp8,sharegpt,32,900,1766.005125,2549.840567,4.202283,14.641521,2277.641572,1557.518150,13.797215,513.391369,161.240535,112.886414
5,ml.g7e.2xlarge,fp8,sharegpt,64,1200,2330.065089,3235.387149,2.702592,12.211753,2652.710631,2661.697411,23.624300,609.217631,159.856427,112.667780
6,ml.g7e.2xlarge,fp8,sharegpt,128,1500,4442.831105,6140.552907,2.373891,6.795519,4727.810932,2961.723198,26.030176,629.428569,160.500000,113.780374
7,ml.g7e.2xlarge,fp8,sharegpt,256,2000,8978.556632,11001.157805,2.255349,4.969028,9251.709173,2966.684550,26.003912,651.223370,157.603103,114.086086
8,ml.g7e.2xlarge,fp8,sharegpt,512,2400,17544.186206,22393.570748,2.297430,4.928392,17822.621093,2919.958065,25.642840,649.630615,158.148457,113.870309


## 21. Stop the proxy

In [ ]:
stop_proxy()

## 22. Clean up

Delete the endpoint, endpoint configuration, and model to stop incurring charges.

> **Note:** Deleting the endpoint stops billing for the GPU instance. This is a destructive, irreversible action.

In [ ]:
from sagemaker.core.resources import EndpointConfig

# The endpoint config shares the endpoint name when created via ModelBuilder.deploy().
try:
    endpoint_config = EndpointConfig.get(endpoint_config_name=endpoint.endpoint_name)
except Exception as e:
    endpoint_config = None
    print(f"Could not fetch endpoint config: {e}")

endpoint.delete()
print("Deleted endpoint.")

if endpoint_config is not None:
    endpoint_config.delete()
    print("Deleted endpoint config.")

core_model.delete()
print("Deleted model.")

## Summary

This notebook walked through an end-to-end deploy-and-benchmark workflow for **Qwen3-30B-A3B-FP8** on Amazon
SageMaker:

1. Resolved the **DJL LMI** container image (vLLM backend) and configured the FP8 model via environment variables.
2. Built and deployed it with the **SageMaker Python SDK v3 `ModelBuilder`** to a single-GPU `ml.g7e.2xlarge`
   endpoint (FP8, TP=1).
3. Invoked it via the Messages API, the LMI generation schema, Qwen3 thinking-mode control, and streaming.
4. Benchmarked the live endpoint with **NVIDIA AIPerf** over an OpenAI→SageMaker proxy, on a realistic
   **ShareGPT** workload, across a concurrency sweep — producing a throughput / latency table.
5. Cleaned up all resources.

### Where to go next
- **Tune for your SLO:** adjust `OPTION_MAX_ROLLING_BATCH_SIZE`, `OPTION_MAX_MODEL_LEN`, and
  `OPTION_GPU_MEMORY_UTILIZATION`; extend the `CONCURRENCY` ladder to find the saturation knee.
- **Scale out:** add SageMaker autoscaling, or compare instance types (e.g. multi-GPU `ml.g7e.12xlarge`).
- **Compare precisions/datasets:** benchmark BF16 vs FP8, or add your own prompt set via the same `single_turn`
  input-file mechanism.
- **AIPerf reference:** https://github.com/ai-dynamo/aiperf